In [ ]:
!git clone https://github.com/VikaGuryeva/DLS_CLIP.git

In [ ]:
%cd DLS_CLIP

In [ ]:
!pwd

In [ ]:
from google.colab import userdata
userdata.get('GITHUB_TOKEN')

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image


# ---------------------------------------------------------
# Проверяем GPU
# ---------------------------------------------------------

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не обнаружен. "
        "Открой Runtime → Change runtime type "
        "и выбери T4 GPU."
    )

DEVICE = torch.device("cuda")

print("Device:", DEVICE)
print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# /content/styleclip-from-scratch
#     код текущей сессии

# /content/styleclip_runtime
#     временная официальная реализация StyleGAN2

# Google Drive/DLS_CLIP/styleclip_from_scratch
#     модели, изображения и latent-коды

# ---------------------------------------------------------
# Эти две строки нужно изменить
# ---------------------------------------------------------

GITHUB_USERNAME = "VikaGuryeva"
REPOSITORY_NAME = "DLS_CLIP"


# ---------------------------------------------------------
# Пути
# ---------------------------------------------------------

# Здесь будет находиться код в текущей Colab-сессии.
REPO_DIR = Path("/content") / REPOSITORY_NAME

# Временные сторонние библиотеки.
# Они не будут добавлены в твой GitHub.
RUNTIME_DIR = Path("/content/styleclip_runtime")
THIRD_PARTY_DIR = RUNTIME_DIR / "third_party"

# Постоянное хранилище в Google Drive.
STORAGE_DIR = (
    Path("/content/drive/MyDrive")
    / "DLS_CLIP"
    / "styleclip_from_scratch"
)

MODEL_DIR = STORAGE_DIR / "models"
DATA_DIR = STORAGE_DIR / "data" / "generated_faces"


for directory in [
    RUNTIME_DIR,
    THIRD_PARTY_DIR,
    STORAGE_DIR,
    MODEL_DIR,
    DATA_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


print("Repository directory:", REPO_DIR)
print("Persistent storage:", STORAGE_DIR)
print("Models:", MODEL_DIR)
print("Data:", DATA_DIR)

In [ ]:
SAFE_REPOSITORY_URL = (
    f"https://github.com/"
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}.git"
)


if not (REPO_DIR / ".git").exists():
    print("Cloning repository...")

    subprocess.run(
        [
            "git",
            "clone",
            SAFE_REPOSITORY_URL,
            str(REPO_DIR),
        ],
        check=True,
    )
else:
    print("Repository already exists.")

    subprocess.run(
        ["git", "pull", "origin", "main"],
        cwd=REPO_DIR,
        check=True,
    )


os.chdir(REPO_DIR)

print("Current directory:", Path.cwd())

In [ ]:
PROJECT_DIRECTORIES = [
    REPO_DIR / "notebooks",
    REPO_DIR / "src",
    REPO_DIR / "tests",
    REPO_DIR / "configs",
    REPO_DIR / "artifacts" / "source_bank",
]

for directory in PROJECT_DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)


# Чтобы src стал Python-пакетом.
src_init = REPO_DIR / "src" / "__init__.py"

if not src_init.exists():
    src_init.write_text(
        '"""StyleCLIP from-scratch project package."""\n',
        encoding="utf-8",
    )


# Git не сохраняет пустые папки.
for directory in [
    REPO_DIR / "notebooks",
    REPO_DIR / "tests",
    REPO_DIR / "configs",
]:
    gitkeep = directory / ".gitkeep"

    if not gitkeep.exists():
        gitkeep.touch()


print("Project structure created.")

In [ ]:
gitignore_content = """
# Python
__pycache__/
*.py[cod]
*.so

# Jupyter
.ipynb_checkpoints/

# Virtual environments
.venv/
venv/

# Secrets
.env
*.token

# Large model files
*.pkl
*.pt
*.pth
*.ckpt

# Generated data
data/
models/
weights/
outputs/
checkpoints/

# Third-party repositories
third_party/

# OS files
.DS_Store
Thumbs.db
""".strip()


(REPO_DIR / ".gitignore").write_text(
    gitignore_content + "\n",
    encoding="utf-8",
)

print((REPO_DIR / ".gitignore").read_text())

In [ ]:
requirements_content = """
ninja
numpy
pandas
Pillow
matplotlib
""".strip()


(REPO_DIR / "requirements.txt").write_text(
    requirements_content + "\n",
    encoding="utf-8",
)

print((REPO_DIR / "requirements.txt").read_text())

In [ ]:
# ---------------------------------------------------------
# Устанавливаем Ninja для CUDA-операций StyleGAN2
# ---------------------------------------------------------

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "ninja",
    ],
    check=True,
)


# ---------------------------------------------------------
# Клонируем официальную реализацию StyleGAN2
# ---------------------------------------------------------

STYLEGAN_REPO = (
    THIRD_PARTY_DIR
    / "stylegan2-ada-pytorch"
)

if not (STYLEGAN_REPO / "legacy.py").exists():
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "https://github.com/NVlabs/"
            "stylegan2-ada-pytorch.git",
            str(STYLEGAN_REPO),
        ],
        check=True,
    )
else:
    print("StyleGAN2 repository already exists.")


if str(STYLEGAN_REPO) not in sys.path:
    sys.path.insert(0, str(STYLEGAN_REPO))


STYLEGAN_COMMIT = subprocess.check_output(
    [
        "git",
        "-C",
        str(STYLEGAN_REPO),
        "rev-parse",
        "HEAD",
    ],
    text=True,
).strip()


print("StyleGAN2 path:", STYLEGAN_REPO)
print("StyleGAN2 commit:", STYLEGAN_COMMIT)

In [ ]:
MODEL_URL = (
    "https://nvlabs-fi-cdn.nvidia.com/"
    "stylegan2-ada-pytorch/pretrained/"
    "transfer-learning-source-nets/"
    "ffhq-res256-mirror-paper256-noaug.pkl"
)

MODEL_PATH = (
    MODEL_DIR
    / "ffhq_stylegan2_256.pkl"
)


if not MODEL_PATH.exists():
    print("Downloading pretrained StyleGAN2 FFHQ model...")

    try:
        torch.hub.download_url_to_file(
            MODEL_URL,
            str(MODEL_PATH),
            progress=True,
        )

    except Exception as error:
        raise RuntimeError(
            "Не удалось скачать веса StyleGAN2. "
            "Проверь интернет-соединение Colab."
        ) from error

else:
    print("Model already exists:", MODEL_PATH)


def calculate_sha256(path: Path) -> str:
    """Вычисляет SHA256 файла."""

    sha256 = hashlib.sha256()

    with path.open("rb") as file:
        for chunk in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(chunk)

    return sha256.hexdigest()


MODEL_SHA256 = calculate_sha256(MODEL_PATH)


print("Model path:", MODEL_PATH)
print(
    "Model size, MB:",
    round(MODEL_PATH.stat().st_size / 1024**2, 2),
)
print("SHA256:", MODEL_SHA256)

In [ ]:
import legacy


with MODEL_PATH.open("rb") as file:
    network_data = legacy.load_network_pkl(file)


if "G_ema" not in network_data:
    raise KeyError(
        "В файле модели отсутствует G_ema."
    )


G = network_data["G_ema"].to(DEVICE)

G.eval()
G.requires_grad_(False)


for parameter in G.parameters():
    parameter.requires_grad_(False)


print("Generator loaded successfully.")
print()
print("z_dim:", G.z_dim)
print("w_dim:", G.w_dim)
print("num_ws:", G.num_ws)
print("c_dim:", G.c_dim)
print("resolution:", G.img_resolution)
print("channels:", G.img_channels)


assert G.img_resolution == 256
assert not G.training
assert not any(
    parameter.requires_grad
    for parameter in G.parameters()
)


print()
print("Generator is frozen:", True)

In [ ]:
SPLITS = {
    "debug": [
        11,
        29,
        47,
        83,
    ],

    "dev": [
        101,
        137,
        211,
        307,
        401,
        503,
        607,
        719,
    ],

    "test": [
        1009,
        1103,
        1201,
        1307,
        1409,
        1511,
        1601,
        1709,
        1801,
        1907,
        2011,
        2111,
    ],
}


total_count = sum(
    len(seeds)
    for seeds in SPLITS.values()
)

print("Debug:", len(SPLITS["debug"]))
print("Dev:", len(SPLITS["dev"]))
print("Test:", len(SPLITS["test"]))
print("Total:", total_count)

assert total_count == 24


def create_z_from_seed(
    seed: int,
    z_dim: int,
    device: torch.device,
) -> torch.Tensor:
    """Создаёт воспроизводимый latent-вектор z."""

    random_state = np.random.RandomState(seed)

    z_numpy = random_state.randn(
        1,
        z_dim,
    ).astype(np.float32)

    return torch.from_numpy(
        z_numpy
    ).to(device)


def image_tensor_to_numpy(
    image: torch.Tensor,
) -> np.ndarray:
    """
    Переводит изображение StyleGAN2:

    [1, 3, H, W], float, [-1, 1]

    в:

    [H, W, 3], uint8, [0, 255].
    """

    if image.ndim != 4 or image.shape[0] != 1:
        raise ValueError(
            "Ожидался tensor [1, C, H, W], "
            f"получено {tuple(image.shape)}."
        )

    image_uint8 = (
        image
        .detach()
        .permute(0, 2, 3, 1)
        .mul(127.5)
        .add(128)
        .clamp(0, 255)
        .to(torch.uint8)
    )

    return image_uint8[0].cpu().numpy()


@torch.no_grad()
def generate_source_sample(
    seed: int,
) -> tuple[
    torch.Tensor,
    torch.Tensor,
    np.ndarray,
]:
    """
    Возвращает:
    z, W+ и сгенерированное изображение.
    """

    z = create_z_from_seed(
        seed=seed,
        z_dim=G.z_dim,
        device=DEVICE,
    )

    class_condition = torch.zeros(
        [1, G.c_dim],
        device=DEVICE,
    )

    w_plus = G.mapping(
        z,
        class_condition,
        truncation_psi=1.0,
        truncation_cutoff=None,
    )

    image_tensor = G.synthesis(
        w_plus,
        noise_mode="const",
        force_fp32=True,
    )

    image_numpy = image_tensor_to_numpy(
        image_tensor
    )

    return (
        z.detach().cpu(),
        w_plus.detach().cpu(),
        image_numpy,
    )

In [ ]:
LATENT_BANK_PATH = (
    DATA_DIR
    / "latent_bank.pt"
)

METADATA_PATH = (
    DATA_DIR
    / "metadata.csv"
)

MODEL_INFO_PATH = (
    DATA_DIR
    / "model_info.json"
)

OVERWRITE = False


def create_latent_bank():
    latent_items = {}
    metadata_rows = []

    for split_name, seeds in SPLITS.items():
        split_directory = (
            DATA_DIR
            / split_name
        )

        split_directory.mkdir(
            parents=True,
            exist_ok=True,
        )

        print(
            f"\nGenerating split: {split_name}"
        )

        for seed in seeds:
            image_id = (
                f"{split_name}_seed_{seed:04d}"
            )

            image_path = (
                split_directory
                / f"{image_id}.png"
            )

            z, w_plus, image_numpy = (
                generate_source_sample(seed)
            )

            Image.fromarray(
                image_numpy
            ).save(image_path)

            latent_items[image_id] = {
                "seed": int(seed),
                "split": split_name,
                "z": z,
                "w_plus": w_plus,
            }

            metadata_rows.append(
                {
                    "image_id": image_id,
                    "split": split_name,
                    "seed": seed,
                    "image_path": str(
                        image_path.relative_to(
                            STORAGE_DIR
                        )
                    ),
                    "latent_key": image_id,
                    "z_shape": str(
                        list(z.shape)
                    ),
                    "w_plus_shape": str(
                        list(w_plus.shape)
                    ),
                    "resolution": (
                        G.img_resolution
                    ),
                    "truncation_psi": 1.0,
                    "noise_mode": "const",
                }
            )

            print(
                f"  {image_id}: "
                f"z={tuple(z.shape)}, "
                f"W+={tuple(w_plus.shape)}"
            )

    latent_bank = {
        "format_version": 1,

        "generator": {
            "model_file": MODEL_PATH.name,
            "model_sha256": MODEL_SHA256,
            "repository_commit": (
                STYLEGAN_COMMIT
            ),
            "z_dim": G.z_dim,
            "w_dim": G.w_dim,
            "num_ws": G.num_ws,
            "c_dim": G.c_dim,
            "resolution": G.img_resolution,
            "channels": G.img_channels,
            "truncation_psi": 1.0,
            "noise_mode": "const",
        },

        "splits": SPLITS,
        "items": latent_items,
    }

    metadata = pd.DataFrame(
        metadata_rows
    )

    return latent_bank, metadata


if (
    LATENT_BANK_PATH.exists()
    and METADATA_PATH.exists()
    and not OVERWRITE
):
    print(
        "Latent bank already exists. "
        "Loading saved data."
    )

    latent_bank = torch.load(
        LATENT_BANK_PATH,
        map_location="cpu",
        weights_only=False,
    )

    metadata = pd.read_csv(
        METADATA_PATH
    )

else:
    latent_bank, metadata = (
        create_latent_bank()
    )

    torch.save(
        latent_bank,
        LATENT_BANK_PATH,
    )

    metadata.to_csv(
        METADATA_PATH,
        index=False,
    )

    model_info = {
        **latent_bank["generator"],
        "pytorch_version": (
            torch.__version__
        ),
        "cuda_version": (
            torch.version.cuda
        ),
        "gpu": (
            torch.cuda.get_device_name(0)
        ),
    }

    with MODEL_INFO_PATH.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            model_info,
            file,
            indent=2,
            ensure_ascii=False,
        )

    print("\nData saved.")


print()
print("Latent bank:", LATENT_BANK_PATH)
print("Metadata:", METADATA_PATH)
print("Model info:", MODEL_INFO_PATH)
print("Number of records:", len(metadata))

metadata.head()

In [ ]:
debug_metadata = (
    metadata[
        metadata["split"] == "debug"
    ]
    .sort_values("seed")
    .reset_index(drop=True)
)


figure, axes = plt.subplots(
    1,
    len(debug_metadata),
    figsize=(14, 4),
)


for axis, row in zip(
    axes,
    debug_metadata.itertuples(),
):
    image_path = (
        STORAGE_DIR
        / row.image_path
    )

    image = Image.open(
        image_path
    ).convert("RGB")

    axis.imshow(image)
    axis.set_title(
        f"seed = {row.seed}"
    )
    axis.axis("off")


plt.tight_layout()
plt.show()

In [ ]:
ARTIFACT_DIR = (
    REPO_DIR
    / "artifacts"
    / "source_bank"
)

ARTIFACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


shutil.copy2(
    METADATA_PATH,
    ARTIFACT_DIR / "metadata.csv",
)

shutil.copy2(
    MODEL_INFO_PATH,
    ARTIFACT_DIR / "model_info.json",
)


print("Copied:")
print(
    ARTIFACT_DIR / "metadata.csv"
)
print(
    ARTIFACT_DIR / "model_info.json"
)

In [ ]:
GIT_NAME = GITHUB_USERNAME
GIT_EMAIL = "GurevaVS@yandex.ru"


subprocess.run(
    [
        "git",
        "config",
        "user.name",
        GIT_NAME,
    ],
    cwd=REPO_DIR,
    check=True,
)

subprocess.run(
    [
        "git",
        "config",
        "user.email",
        GIT_EMAIL,
    ],
    cwd=REPO_DIR,
    check=True,
)


print("Git identity configured.")

In [ ]:
from google.colab import userdata


github_token = userdata.get(
    "GITHUB_TOKEN"
)

if not github_token:
    raise RuntimeError(
        "Secret GITHUB_TOKEN не найден. "
        "Добавь его через панель Secrets."
    )


authenticated_url = (
    f"https://{GITHUB_USERNAME}:"
    f"{github_token}@github.com/"
    f"{GITHUB_USERNAME}/"
    f"{REPOSITORY_NAME}.git"
)


# Временно устанавливаем URL с авторизацией.
subprocess.run(
    [
        "git",
        "remote",
        "set-url",
        "origin",
        authenticated_url,
    ],
    cwd=REPO_DIR,
    check=True,
)


try:
    subprocess.run(
        ["git", "add", "."],
        cwd=REPO_DIR,
        check=True,
    )

    status = subprocess.run(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        check=True,
        capture_output=True,
        text=True,
    )

    if status.stdout.strip():
        subprocess.run(
            [
                "git",
                "commit",
                "-m",
                "Set up project and source latent bank",
            ],
            cwd=REPO_DIR,
            check=True,
        )

        subprocess.run(
            [
                "git",
                "push",
                "origin",
                "main",
            ],
            cwd=REPO_DIR,
            check=True,
        )

        print("Changes pushed to GitHub.")

    else:
        print("No changes to commit.")

finally:
    # Удаляем токен из remote URL.
    subprocess.run(
        [
            "git",
            "remote",
            "set-url",
            "origin",
            SAFE_REPOSITORY_URL,
        ],
        cwd=REPO_DIR,
        check=True,
    )

    del github_token
    del authenticated_url

In [ ]:
from pathlib import Path
import shutil

# Название текущего notebook
NOTEBOOK_NAME = "01_generate_source_faces.ipynb"

# Папка клонированного GitHub-репозитория
REPO_DIR = Path("/content/DLS_CLIP")

# Стандартная папка Colab в Google Drive
DRIVE_NOTEBOOK_DIR = (
    Path("/content/drive/MyDrive")
    / "Colab Notebooks"
)

# Куда положить notebook внутри репозитория
NOTEBOOK_TARGET_DIR = REPO_DIR / "notebooks"
NOTEBOOK_TARGET_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Сначала проверяем стандартный путь
notebook_source = (
    DRIVE_NOTEBOOK_DIR
    / NOTEBOOK_NAME
)

# Если там файла нет, ищем по всему MyDrive
if not notebook_source.exists():
    print(
        "Файл не найден в стандартной папке. "
        "Ищу по Google Drive..."
    )

    found_paths = list(
        Path("/content/drive/MyDrive").rglob(
            NOTEBOOK_NAME
        )
    )

    if len(found_paths) == 0:
        raise FileNotFoundError(
            f"Notebook {NOTEBOOK_NAME} не найден.\n"
            "Нажми Ctrl + S, подожди несколько секунд "
            "и запусти эту ячейку снова."
        )

    print("Найдены файлы:")

    for path in found_paths:
        print(path)

    # Если найдено несколько копий,
    # берём файл с самым поздним временем изменения
    notebook_source = max(
        found_paths,
        key=lambda path: path.stat().st_mtime,
    )

notebook_target = (
    NOTEBOOK_TARGET_DIR
    / NOTEBOOK_NAME
)

shutil.copy2(
    notebook_source,
    notebook_target,
)

print()
print("Исходный файл:")
print(notebook_source)

print()
print("Скопирован в репозиторий:")
print(notebook_target)

print()
print(
    "Размер:",
    notebook_target.stat().st_size,
    "байт",
)